# Transformer Triton kernel — Colab runner

Use a GPU runtime. This notebook invokes the full contract suite, untouched organizer harnesses, fail-closed performance matrix, and profiler used by local validation. Colab results describe the assigned Colab GPU, not the curated RTX 5070 Ti run.

## 1. Clone the complete repository

Individual-file upload is insufficient because the kernel, dispatcher, manifests, and tools are separate modules. The next cell checks out the flagship submission branch, first trying anonymous access. While the repository remains private, it securely prompts for a short-lived fine-grained GitHub token with read-only Contents access. The token is passed through a temporary `GIT_ASKPASS` helper, is not stored in the clone URL, and should be revoked after the run. The implementation fingerprint is checked before any test or benchmark runs. Restart the runtime before rerunning this cell against a different branch or revision.

In [ ]:
import getpass
import os
import stat
import subprocess
import tempfile

repo_dir = '/content/tiktok-techjam-2026'
repo_url = 'https://github.com/lukeai-tan/tiktok-techjam-2026.git'
repo_ref = 'feat/transformer-gpu-kernel-implementation'
expected_implementation_sha256 = '9159177a21d039366ed4d3aef431b4b14d3bcef26d5eeaab0808efa739294029'
clone_command = ['git', 'clone', '--branch', repo_ref, '--single-branch', repo_url, repo_dir]
if not os.path.isdir(repo_dir):
    public_access = subprocess.run(
        ['git', 'ls-remote', repo_url, f'refs/heads/{repo_ref}'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    ).returncode == 0
    if public_access:
        subprocess.run(clone_command, check=True)
    else:
        token = getpass.getpass('Fine-grained GitHub token (Contents: read-only): ')
        askpass_path = None
        clone_env = None
        try:
            with tempfile.NamedTemporaryFile('w', suffix='.sh', delete=False) as askpass:
                askpass.write("""#!/bin/sh
case "$1" in
  *Username*) printf '%s\n' 'x-access-token' ;;
  *) printf '%s\n' "$TIKTOK_TECHJAM_GITHUB_TOKEN" ;;
esac
""")
                askpass_path = askpass.name
            os.chmod(askpass_path, stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)
            clone_env = os.environ.copy()
            clone_env.update({
                'GIT_ASKPASS': askpass_path,
                'GIT_TERMINAL_PROMPT': '0',
                'TIKTOK_TECHJAM_GITHUB_TOKEN': token,
            })
            subprocess.run(
                clone_command,
                check=True,
                env=clone_env,
            )
        finally:
            token = None
            clone_env = None
            if askpass_path and os.path.exists(askpass_path):
                os.remove(askpass_path)
else:
    current_ref = subprocess.check_output(
        ['git', '-C', repo_dir, 'branch', '--show-current'], text=True
    ).strip()
    if current_ref != repo_ref:
        raise RuntimeError(
            f'Existing clone is on {current_ref!r}, expected {repo_ref!r}; restart the runtime.'
        )
    dirty_paths = subprocess.check_output(
        ['git', '-C', repo_dir, 'status', '--porcelain'], text=True
    ).strip()
    if dirty_paths:
        raise RuntimeError('Existing clone has local changes; restart the runtime for a clean run.')
os.chdir(repo_dir)
print(os.getcwd())
print('ref', repo_ref, 'commit', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## 2. Install test dependencies and verify CUDA, PyTorch, and Triton

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy==2.5.2', 'pytest==9.1.1'], check=True)
import torch, triton
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU'
print('torch', torch.__version__, 'cuda', torch.version.cuda)
print('triton', triton.__version__)
print('gpu', torch.cuda.get_device_name(), 'capability', torch.cuda.get_device_capability())
from tools.capture_environment import implementation_fingerprint
actual_implementation_sha256, implementation_paths = implementation_fingerprint()
assert actual_implementation_sha256 == expected_implementation_sha256, (
    f'Implementation fingerprint {actual_implementation_sha256} does not match the flagship '
    f'{expected_implementation_sha256}; restart the runtime and clone the current branch.'
)
print('implementation_sha256', actual_implementation_sha256)
print('fingerprinted_paths', len(implementation_paths))

## 3. Run the CPU/GPU contract suite

A compiler and Python headers may be required when Colab first builds Triton's driver shim.

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q'], check=True)

## 4. Untouched organizer PyTorch harness

This loads the checksum-frozen organizer file and injects only the submitted `UserOptimizedTransformer` extension point.

In [ ]:
subprocess.run([sys.executable, 'benchmarks/run_organizer_torch.py', '--device', 'cuda', '--evidence-out', 'results/colab-organizer-default.json'], check=True)

## 5. Published final evaluator matrix

This is the primary 14-row organizer-published shape table. All 13 executable rows must pass strict accuracy; the exact source-authorized 100,000-token stress row is reported as a resource skip and never counted as a pass.

In [ ]:
subprocess.run([sys.executable, 'benchmarks/run_organizer_validation.py', '--matrix', 'benchmarks/final_evaluator_shapes.json', '--device', 'cuda', '--out', 'results/colab-published-final.json'], check=True)

## 6. Rigorous source-derived validation

Every feasible shape signaled by both supplied files runs through the untouched selected PyTorch harness. The source-designated 100,000-token resource skip is reported separately and never counted as a pass.

In [ ]:
subprocess.run([sys.executable, 'benchmarks/run_organizer_validation.py', '--device', 'cuda', '--out', 'results/colab-source-derived.json'], check=True)

## 7. Project held-out matrix

This fails unless every requested case is PASS. It never converts compilation errors or OOM-only runs into success.

In [ ]:
subprocess.run([sys.executable, 'benchmarks/run_matrix.py', '--device', 'cuda', '--attention-backend', 'auto', '--accuracy-trials', '5', '--out', 'results/colab-matrix.json'], check=True)

## 8. Profiler proof

Profile both the flagship's strongest published row and a held-out causal-plus-padding case. The JSON summaries and Chrome trace files prove which attention backend executed.

In [ ]:
subprocess.run([sys.executable, 'benchmarks/profile_cases.py', '--manifest', 'benchmarks/campaign5_profile_shapes.json', '--case', 'final-06-b10000-d128-h4-s128', '--dtype', 'float32', '--attention-backend', 'auto', '--steps', '5', '--out', 'results/colab-final-row6-profile.json', '--trace', 'results/colab-final-row6-trace.json'], check=True)
subprocess.run([sys.executable, 'benchmarks/profile_cases.py', '--manifest', 'benchmarks/campaign5_profile_shapes.json', '--case', 'final-07-b64-d32-h4-s128', '--dtype', 'float32', '--attention-backend', 'auto', '--steps', '5', '--out', 'results/colab-final-row7-profile.json', '--trace', 'results/colab-final-row7-trace.json'], check=True)
subprocess.run([sys.executable, 'benchmarks/profile_cases.py', '--manifest', 'benchmarks/campaign4_profile_shapes.json', '--case', 'final-11-b64-d128-h16-s128', '--dtype', 'float32', '--attention-backend', 'auto', '--steps', '5', '--out', 'results/colab-final-row11-profile.json', '--trace', 'results/colab-final-row11-trace.json'], check=True)
subprocess.run([sys.executable, 'benchmarks/profile_cases.py', '--manifest', 'benchmarks/campaign5_profile_shapes.json', '--case', 'long-causal-padding', '--dtype', 'float32', '--attention-backend', 'auto', '--steps', '5', '--out', 'results/colab-heldout-profile.json', '--trace', 'results/colab-heldout-trace.json'], check=True)

## 9. Download one evidence bundle

In [ ]:
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile
from google.colab import files

evidence_paths = [
    Path('results/colab-organizer-default.json'),
    Path('results/colab-published-final.json'),
    Path('results/colab-source-derived.json'),
    Path('results/colab-matrix.json'),
    Path('results/colab-final-row6-profile.json'),
    Path('results/colab-final-row6-trace.json'),
    Path('results/colab-final-row7-profile.json'),
    Path('results/colab-final-row7-trace.json'),
    Path('results/colab-final-row11-profile.json'),
    Path('results/colab-final-row11-trace.json'),
    Path('results/colab-heldout-profile.json'),
    Path('results/colab-heldout-trace.json'),
]
missing = [str(path) for path in evidence_paths if not path.is_file()]
assert not missing, f'Missing evidence: {missing}'
bundle_path = Path('results/colab-evidence.zip')
with ZipFile(bundle_path, 'w', compression=ZIP_DEFLATED) as archive:
    for path in evidence_paths:
        archive.write(path, arcname=path.name)
files.download(str(bundle_path))